In [76]:
import pandas as pd
import numpy as np

In [77]:
train_data = pd.read_csv(
    "data/fraudTrain.csv", 
    index_col=0, 
    parse_dates=["trans_date_trans_time", "dob"]
)

test_data = pd.read_csv(
    "data/fraudTest.csv", 
    index_col=0, 
    parse_dates=["trans_date_trans_time", "dob"]
)

train_data.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [78]:
#Add age feature
def add_age(df):
    df = df.copy()
    df["age"] = (df["trans_date_trans_time"] - df["dob"]).dt.days // 365.25
    return df

train_data = add_age(train_data)
test_data = add_age(test_data)

train_data[["dob", "trans_date_trans_time", "age"]].head()
    

,dob,trans_date_trans_time,age
0,1988-03-09,2019-01-01 00:00:18,30.0
1,1978-06-21,2019-01-01 00:00:44,40.0
2,1962-01-19,2019-01-01 00:00:51,56.0
3,1967-01-12,2019-01-01 00:01:16,51.0
4,1986-03-28,2019-01-01 00:03:06,32.0


In [79]:
def time_features(df):
    df = df.copy()
    df["hour"] = df["trans_date_trans_time"].dt.hour
    df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
    df["is_night"] = df["hour"].isin(range(22,24)) | df["hour"].isin(range(0,4))
    return df

train_data = time_features(train_data)
test_data = time_features(test_data)

train_data[["trans_date_trans_time", "hour", "day_of_week", "is_night"]].head()

,trans_date_trans_time,hour,day_of_week,is_night
0,2019-01-01 00:00:18,0,1,True
1,2019-01-01 00:00:44,0,1,True
2,2019-01-01 00:00:51,0,1,True
3,2019-01-01 00:01:16,0,1,True
4,2019-01-01 00:03:06,0,1,True


In [80]:
print(train_data["category"].nunique(), "unique categories")

train_data = pd.get_dummies(train_data, columns=["category"], prefix="category", dtype=int)
test_data = pd.get_dummies(test_data, columns=["category"], prefix="category")

train_data.head()


14 unique categories


,trans_date_trans_time,cc_num,merchant,amt,first,last,gender,street,city,state,...,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,...,0,0,0,0,1,0,0,0,0,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,...,1,0,0,0,0,0,0,0,0,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,...,0,0,0,0,0,0,0,0,0,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,...,0,0,0,0,0,0,0,0,0,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,...,0,0,0,0,0,1,0,0,0,0


In [81]:
#Apply frequency encoding to high cardinality categorical features

def frequency_encode(train_df, test_df, col):
    freq_map = train_df[col].value_counts(normalize=True)
    train_df[col + "_freq"] = train_df[col].map(freq_map)
    test_df[col + "_freq"] = test_df[col].map(freq_map).fillna(0)  # unseen values in test -> 0
    return train_df, test_df

for col in ["merchant", "city", "job"]:
    train_data, test_data = frequency_encode(train_data, test_data, col)

train_data[["merchant", "merchant_freq", "city", "city_freq", "job", "job_freq"]].head()

,merchant,merchant_freq,city,city_freq,job,job_freq
0,"fraud_Rippin, Kub and Mann",0.000977,Moravian Falls,0.001564,"Psychologist, counselling",0.002734
1,"fraud_Heller, Gutmann and Zieme",0.001930,Orient,0.002734,Special educational needs teacher,0.003932
2,fraud_Lind-Buckridge,0.001461,Malad City,0.000388,Nature conservation officer,0.000394
3,"fraud_Kutch, Hermiston and Farrell",0.002015,Boulder,0.000380,Patent attorney,0.001951
4,fraud_Keeling-Crist,0.001228,Doe Hill,0.001556,Dance movement psychotherapist,0.001556


In [82]:
train_data = train_data.drop(
    columns=["trans_date_trans_time", "cc_num", "merchant", "first",
     "last", "street", "city", "state", "zip", "lat", "long", 
     "job", "dob", "trans_num", "unix_time", "merch_lat", "merch_long"]
)

test_data = test_data.drop(
    columns=["trans_date_trans_time", "cc_num", "merchant", "first",
     "last", "street", "city", "state", "zip", "lat", "long", 
     "job", "dob", "trans_num", "unix_time", "merch_lat", "merch_long"]
)

print(train_data.shape, test_data.shape)


(1296675, 25) (555719, 25)


In [83]:
train_data.head()

,amt,gender,city_pop,is_fraud,age,hour,day_of_week,is_night,category_entertainment,category_food_dining,...,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,merchant_freq,city_freq,job_freq
0,4.97,F,3495,0,30.0,0,1,True,0,0,...,0,1,0,0,0,0,0,0.000977,0.001564,0.002734
1,107.23,F,149,0,40.0,0,1,True,0,0,...,0,0,0,0,0,0,0,0.001930,0.002734,0.003932
2,220.11,M,4154,0,56.0,0,1,True,1,0,...,0,0,0,0,0,0,0,0.001461,0.000388,0.000394
3,45.00,M,1939,0,51.0,0,1,True,0,0,...,0,0,0,0,0,0,0,0.002015,0.000380,0.001951
4,41.96,M,99,0,32.0,0,1,True,0,0,...,0,0,1,0,0,0,0,0.001228,0.001556,0.001556


In [84]:
#Only categorical feature left is gender, map to 0/1 (only 2 classes)

train_data["gender"] = (train_data["gender"] == "M").astype(int)
test_data["gender"] = (test_data["gender"] == "M").astype(int)

train_data["gender"].value_counts() 

gender
0    709863
1    586812
Name: count, dtype: int64

In [85]:
#Final engineered data set
train_data.head()

,amt,gender,city_pop,is_fraud,age,hour,day_of_week,is_night,category_entertainment,category_food_dining,...,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,merchant_freq,city_freq,job_freq
0,4.97,0,3495,0,30.0,0,1,True,0,0,...,0,1,0,0,0,0,0,0.000977,0.001564,0.002734
1,107.23,0,149,0,40.0,0,1,True,0,0,...,0,0,0,0,0,0,0,0.001930,0.002734,0.003932
2,220.11,1,4154,0,56.0,0,1,True,1,0,...,0,0,0,0,0,0,0,0.001461,0.000388,0.000394
3,45.00,1,1939,0,51.0,0,1,True,0,0,...,0,0,0,0,0,0,0,0.002015,0.000380,0.001951
4,41.96,1,99,0,32.0,0,1,True,0,0,...,0,0,1,0,0,0,0,0.001228,0.001556,0.001556


In [86]:
train_data.to_parquet("data/train_data_engineered.parquet")
test_data.to_parquet("data/test_data_engineered.parquet")